In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
import random

# ============================================================
#  Configuration
# ============================================================

# ✅ URL de recherche WordPress (couvre TOUTES les pages avec "accident")
SEARCH_URL = "https://emedia.sn/?s=accident&paged={}"

# ✅ Mots-clés élargis pour le filtre (sécurité supplémentaire)
MOTS_CLES = [
    "accident", "collision", "carambolage",
    "choc", "renversé", "percuté", "crash",
    "mort", "blessé", "victime", "route"
]

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                  "AppleWebKit/537.36 (KHTML, like Gecko) "
                  "Chrome/120.0.0.0 Safari/537.36",
    "Accept-Language": "fr-FR,fr;q=0.9",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
}

MAX_PAGES   = 35       # Nombre max de pages à scraper (ajuste selon tes besoins)
SLEEP_MIN   = 1.5      # Délai min entre requêtes (secondes)
SLEEP_MAX   = 3.0      # Délai max entre requêtes (secondes)
FICHIER_CSV = "accidents_emedia.csv"


# ============================================================
#  Fonctions utilitaires
# ============================================================

def get_page(url: str) -> BeautifulSoup | None:
    """Récupère une page et retourne un objet BeautifulSoup."""
    try:
        response = requests.get(url, headers=HEADERS, timeout=15)
        response.raise_for_status()
        return BeautifulSoup(response.content, "html.parser")
    except requests.RequestException as e:
        print(f"    ⚠️  Erreur requête : {e}")
        return None


def get_nombre_pages(soup: BeautifulSoup) -> int:
    """Détecte automatiquement le nombre total de pages de résultats."""
    # WordPress place souvent la pagination dans .nav-links ou .page-numbers
    pagination = soup.find("div", class_=["nav-links", "pagination", "page-numbers"])
    if pagination:
        pages = pagination.find_all("a", class_="page-numbers")
        numeros = []
        for p in pages:
            try:
                numeros.append(int(p.text.strip()))
            except ValueError:
                pass
        if numeros:
            return max(numeros)
    return MAX_PAGES  # fallback


def extraire_articles_page(soup: BeautifulSoup) -> list[dict]:
    """Extrait les liens et titres d'articles d'une page de résultats."""
    articles = []
    # WordPress utilise souvent h2.entry-title ou h3.entry-title
    selectors = [
        soup.find_all("h2", class_=lambda c: c and "title" in c.lower()),
        soup.find_all("h3", class_=lambda c: c and "title" in c.lower()),
        soup.find_all("h2"),
        soup.find_all("h3"),
    ]

    titres_trouves = []
    for selector in selectors:
        if selector:
            titres_trouves = selector
            break

    for t in titres_trouves:
        a = t.find("a")
        if not a:
            continue
        titre = a.text.strip()
        lien  = a.get("href", "")
        if titre and lien:
            articles.append({"title": titre, "link": lien})

    return articles


def extraire_contenu_article(url: str) -> dict:
    """Va dans un article et extrait date, description complète."""
    soup = get_page(url)
    if not soup:
        return {"date": "", "description": ""}

    # --- Date ---
    date = ""
    date_tag = soup.find("time")
    if date_tag:
        date = date_tag.get("datetime", date_tag.text.strip())

    # Fallback : chercher dans les méta
    if not date:
        meta_date = soup.find("meta", {"property": "article:published_time"})
        if meta_date:
            date = meta_date.get("content", "")

    # --- Contenu / Description ---
    # WordPress place le contenu dans .entry-content ou .post-content
    contenu_div = (
        soup.find("div", class_="entry-content") or
        soup.find("div", class_="post-content") or
        soup.find("article")
    )

    if contenu_div:
        paragraphes = contenu_div.find_all("p")
    else:
        paragraphes = soup.find_all("p")

    description = " ".join(
        p.text.strip() for p in paragraphes if len(p.text.strip()) > 30
    )

    return {"date": date, "description": description}


# ============================================================
#  Programme principal
# ============================================================

def main():
    print("=" * 60)
    print("  Scraper emedia.sn — Recherche : 'accident'")
    print("=" * 60)

    # --- Étape 1 : Détecter le nombre de pages ---
    print("\n📄 Détection du nombre de pages...")
    soup_page1 = get_page(SEARCH_URL.format(1))
    if not soup_page1:
        print("❌ Impossible d'accéder au site. Vérifie ta connexion.")
        return

    nb_pages = min(get_nombre_pages(soup_page1), MAX_PAGES)
    print(f"   → {nb_pages} pages détectées (max configuré : {MAX_PAGES})")

    # --- Étape 2 : Collecter tous les liens d'articles ---
    print(f"\n🔗 Collecte des liens sur {nb_pages} pages...")
    tous_liens = []

    # Traiter la page 1 déjà chargée
    articles_p1 = extraire_articles_page(soup_page1)
    tous_liens.extend(articles_p1)
    print(f"   Page 1 : {len(articles_p1)} articles trouvés")

    for page in range(2, nb_pages + 1):
        url = SEARCH_URL.format(page)
        soup = get_page(url)
        if not soup:
            continue

        articles_page = extraire_articles_page(soup)

        # Vérifier si la page est vide (fin de la pagination)
        if not articles_page:
            print(f"   Page {page} : vide → arrêt")
            break

        tous_liens.extend(articles_page)
        print(f"   Page {page} : {len(articles_page)} articles trouvés")
        time.sleep(random.uniform(SLEEP_MIN, SLEEP_MAX))

    # Dédoublonner par lien
    seen = set()
    liens_uniques = []
    for art in tous_liens:
        if art["link"] not in seen:
            seen.add(art["link"])
            liens_uniques.append(art)

    print(f"\n✅ Total liens uniques : {len(liens_uniques)}")

    # --- Étape 3 : Scraper chaque article ---
    print(f"\n📰 Scraping du contenu de chaque article...\n")
    resultats = []

    for i, art in enumerate(liens_uniques, 1):
        titre = art["title"]
        lien  = art["link"]

        # Filtrer par mots-clés (sécurité supplémentaire)
        if not any(mot in titre.lower() for mot in MOTS_CLES):
            continue

        print(f"  [{i}/{len(liens_uniques)}] {titre[:65]}...")
        contenu = extraire_contenu_article(lien)

        resultats.append({
            "date":        contenu["date"],
            "titre":       titre,
            "description": contenu["description"],
            "lien":        lien,
        })

        time.sleep(random.uniform(SLEEP_MIN, SLEEP_MAX))

    # --- Étape 4 : Sauvegarder en CSV ---
    df = pd.DataFrame(resultats)

    if df.empty:
        print("\n⚠️  Aucun article extrait. Vérifie les sélecteurs HTML.")
        return

    # Nettoyage basique
    df["date"]        = df["date"].str.strip()
    df["titre"]       = df["titre"].str.strip()
    df["description"] = df["description"].str.replace(r"\s+", " ", regex=True).str.strip()

    df.to_csv(FICHIER_CSV, index=False, encoding="utf-8-sig")

    print("\n" + "=" * 60)
    print(f"  ✅  Fichier créé     : {FICHIER_CSV}")
    print(f"  📰  Articles extraits : {len(df)}")
    print(f"  📄  Pages scrapées   : {nb_pages}")
    print("=" * 60)
    print(df[["date", "titre"]].head(10).to_string(index=False))


if __name__ == "__main__":
    main()

In [ ]:
import pandas as pd
df = pd.read_csv("accidents_emedia.csv")

In [ ]:
print(df.columns)
print(df.head())
print(len(df))

In [ ]:
def extract_city(text):
    if not isinstance(text, str):
        return "Inconnue"
    cities = ["Dakar", "Thiès", "Diourbel", "Kaolack", "Saint-Louis", "Ziguinchor", "Tambacounda", "Kédougou", "Kolda", "Matam", "Louga", "Fatick", "Sédhiou", "Casamance", "Bignona"]

    for city in cities:
        if city.lower() in text.lower():
            return city
    return "Inconnue"

# Extraire d'abord de la description
df["ville"] = df["description"].apply(extract_city)

# Pour les lignes où la ville est "Inconnue", essayer le titre
mask = df["ville"] == "Inconnue"
df.loc[mask, "ville"] = df.loc[mask, "titre"].apply(extract_city)

In [ ]:
df.groupby("ville").size().sort_values(ascending=False)

In [ ]:
df['titre']

In [ ]:
# Dictionnaire complet mis à jour avec les nouveaux indices implicites
mapping_contextuel = {
    "Dakar": [
        "TER", "Vdn", "Corniche", "Pikine", "Guédiawaye", "Rufisque", "Diamniadio", 
        "Colobane", "Fann", "Almadies", "Grand Yoff", "Parcelles Assainies", 
        "Patte d'Oie", "Sébikotane", "Bargny", "Keur Massar", "Malika", "Yoff",
        "Rond-point EMG", "Bus Tata", "Cité imbécile"
    ],
    "Thiès": [
        "Mbour", "Saly", "Tivaouane", "Pout", "Joal", "Sindia", "Somone", 
        "Ndiass", "AIBD", "Khombole", "Meckhe"
    ],
    "Diourbel": [
        "Touba", "Mbacké", "Bambey"
    ],
    "Saint-Louis": [
        "Podor", "Richard-Toll", "Dagana", "Ndioum", "Ross Béthio", "Pété"
    ],
    "Ziguinchor": [
        "Bignona", "Oussouye", "Casamance", "Sindian", "Kafountine", "Blouf", "Mlomp", "Ediamath"
    ],
    "Kaolack": [
        "Nioro", "Passy", "Guinguinéo", "Kahone"
    ],
    "Fatick": [
        "Foundiougne", "Gossas", "Sokone", "Toubacouta"
    ],
    "Louga": [
        "Kébémer", "Linguère", "Dahra"
    ],
    "Kaffrine": [
        "Kaffrine", "Boulel", "Mbirkilane", "Sagna", "Koungheul", "Wengui"
    ],
    "Tambacounda": [
        "Goudiry", "Bakel", "Koupentoum", "DCSOM"
    ],
    "Matam": [
        "Ourossogui", "Kanel", "Ranérou", "Seddo Sebbé"
    ],
    "Sédhiou": [
        "Dianka"
    ]
}

In [ ]:
def deep_extract_city(row):
    # Si la ville est déjà trouvée, on la garde
    if row["ville"] != "Inconnue":
        return row["ville"]
    
    # On combine titre et description pour la recherche
    texte_complet = f"{str(row['titre'])} {str(row['description'])}".lower()
    
    # Parcours du dictionnaire contextuel
    for ville, mots_cles in mapping_contextuel.items():
        for mot in mots_cles:
            if mot.lower() in texte_complet:
                return ville
                
    return "Inconnue"

# Application de la fonction
df["ville"] = df.apply(deep_extract_city, axis=1)

In [ ]:
df.groupby("ville").size().sort_values(ascending=False)

In [ ]:
# Mots-clés qui indiquent un accident de la route
mots_routiers = [
    "collision", "percuté", "renversé", "choc", "dérapage", "tonneau", 
    "bus", "car", "moto", "véhicule", "camion", "scooter", "minibus", 
    "ndaga ndiaye", "sept-place", "tata", "clando", "tipper", "poids lourd"
]

# Mots-clés à exclure (Bruit)
mots_exclusion = [
    "pirogue", "chavirement", "mer", "noyade", "mortier", "rebelle", 
    # "mfdc", "armée", "militaire", "élection", "manifestation"
]

In [ ]:
def filtrer_accidents_route(df):
    # 1. Mise en minuscule pour la comparaison
    df['texte_recherche'] = (df['titre'] + " " + df['description']).str.lower()
    
    # 2. On garde si un mot routier est présent
    masque_routier = df['texte_recherche'].apply(
        lambda x: any(mot in str(x) for mot in mots_routiers)
    )
    
    # 3. On exclut si un mot banni est présent (prioritaire)
    masque_exclusion = df['texte_recherche'].apply(
        lambda x: any(mot in str(x) for mot in mots_exclusion)
    )
    
    # Application du filtre : Routier ET NON Exclusion
    df_clean = df[masque_routier & ~masque_exclusion].copy()
    
    # Supprimer la colonne temporaire
    df_clean.drop(columns=['texte_recherche'], inplace=True)
    
    return df_clean

# Utilisation
df_accidents_route = filtrer_accidents_route(df)

print(f"Nombre d'articles avant : {len(df)}")
print(f"Nombre d'accidents routiers après filtrage : {len(df_accidents_route)}")

In [ ]:
df_accidents_route['date'] = pd.to_datetime(df_accidents_route['date'])

In [ ]:
df_accidents_route.groupby("ville").size().sort_values(ascending=False)

In [ ]:
# Filtrer pour n'avoir que les villes inconnues
df_inconnues = df_accidents_route[df_accidents_route["ville"] == "Inconnue"].copy()

# Afficher les premières lignes pour inspection (titre et description)
print(f"Il reste {len(df_inconnues)} lignes à identifier.")
pd.set_option('display.max_colwidth', 150) # Pour lire plus facilement les descriptions
print(df_inconnues[["titre", "description"]].head(20))

In [ ]:
# A supprimer après vérification manuelle
import re

def extract_gravite(text):
    if not isinstance(text, str):
        return 0, 0
    
    text = text.lower()
    # Dictionnaire pour convertir les petits nombres en lettres
    nombre_map = {
        "un": 1, "une": 1, "deux": 2, "trois": 3, "quatre": 4, 
        "cinq": 5, "six": 6, "sept": 7, "huit": 8, "neuf": 9, "dix": 10
    }
    
    def get_number(match_group):
        val = match_group.strip()
        if val.isdigit():
            return int(val)
        return nombre_map.get(val, 0)

    # Extraction des morts/décès
    morts = 0
    pattern_morts = r"(\d+|un|une|deux|trois|quatre|cinq|six|sept|huit|neuf|dix)\s*(mort|décès|vie)"
    match_morts = re.search(pattern_morts, text)
    if match_morts:
        morts = get_number(match_morts.group(1))
    
    # Extraction des blessés
    blesses = 0
    pattern_blesses = r"(\d+|un|une|deux|trois|quatre|cinq|six|sept|huit|neuf|dix)\s*(blessé|victime)"
    match_blesses = re.search(pattern_blesses, text)
    if match_blesses:
        blesses = get_number(match_blesses.group(1))
        
    return morts, blesses

# Application au DataFrame
# On combine titre et description pour maximiser les chances de trouver l'info
df_accidents_route['temp_gravite'] = (df_accidents_route['titre'] + " " + df_accidents_route['description']).apply(extract_gravite)

# Séparation en deux colonnes distinctes
df_accidents_route[['nb_morts', 'nb_blesses']] = pd.DataFrame(df_accidents_route['temp_gravite'].tolist(), index=df_accidents_route.index)
df_accidents_route.drop(columns=['temp_gravite'], inplace=True)

In [ ]:
import re

def extract_gravite_v3(row):
    # On nettoie le texte pour faciliter la recherche
    text = f"{str(row['titre'])} {str(row['description'])}".lower()
    
    # 1. Dictionnaire des quantités
    quantites = {
        "une vingtaine": 20, "une dizaine": 10, "une quinzaine": 15,
        "six": 6, "sept": 7, "quatre": 4, "trois": 3, "deux": 2, "un": 1, "une": 1
    }

    morts, blesses = 0, 0

    # 2. Extraction des Blessés (Priorité à la ligne 8)
    # On cherche d'abord les nombres (chiffres ou mots) suivis de "blessé"
    for mot, valeur in quantites.items():
        if re.search(rf"({valeur}|{mot})\s*(blessés?|graves?)", text):
            blesses = max(blesses, valeur)
    
    # On cherche aussi les chiffres purs (ex: "15 blessés")
    chiffres_blesses = re.findall(r"(\d+)\s*(blessés?)", text)
    for val, _ in chiffres_blesses:
        blesses = max(blesses, int(val))

    # 3. Extraction des Morts (Avec filtre anti-faux positifs)
    # On ne compte un mort "par défaut" que si c'est un récit d'accident réel
    mots_action = ["tue", "meurt", "perd la vie", "perdent la vie", "coûte la vie", "décédé", "fauché"]
    
    for mot, valeur in quantites.items():
        if re.search(rf"({valeur}|{mot})\s*(morts?|décès|vie|tués?)", text):
            morts = max(morts, valeur)
            
    if morts == 0 and any(exp in text for exp in mots_action):
        # Si on ne trouve pas de chiffre mais qu'un mot d'action est là : c'est 1 mort
        morts = 1

    return pd.Series([morts, blesses])

# Application
df_accidents_route[['nb_morts', 'nb_blesses']] = df_accidents_route.apply(extract_gravite_v3, axis=1)

In [ ]:
# 1. Supprimer les lignes où la ville est encore "Inconnue" (comme le Mali)
df_final = df[df["ville"] != "Inconnue"].copy()

# 2. Supprimer les doublons potentiels (si deux articles parlent du même accident)
df_final = df_final.drop_duplicates(subset=['titre'])

print(f"Ton dataset contient maintenant {len(df_final)} accidents qualifiés avec localisation et gravité.")

In [ ]:
# Vérifier les lignes où on a détecté des morts
print(df_accidents_route[df_accidents_route['nb_morts'] > 0][['titre', 'nb_morts', 'nb_blesses']].head())

In [ ]:
df_accidents_route.head(20)

In [ ]:
# Suppression des lignes "Bruit"
mots_bruit = ["espagne", "contrôle technique", "lutter contre", "prévention", "ministre rappelle"]
df_final = df_accidents_route[~df_accidents_route['titre'].str.lower().str.contains('|'.join(mots_bruit))]

# Suppression des morts à l'étranger (si ville identifiée est hors Sénégal)
# Tu peux ajouter "Espagne" ou "Mali" dans ton dictionnaire pour les filtrer ensuite.

In [ ]:
df_final.head(10)

In [ ]:
# Exporter les données finales dans le fichier accidents_senegal.csv
df_final.to_csv("accidents_senegal.csv", index=False, encoding="utf-8-sig")

In [ ]:
import pandas as pd
# lire le fichier accidents_senegal.csv
df_l = pd.read_csv("accidents_senegal.csv", on_bad_lines='skip')

In [ ]:
# ajouter une clonne source
df_final["source"] = "emedia.sn"

In [ ]:
df_final.columns.tolist()

In [ ]:
# affciher les élement uniques de la colonne source
print(df_final["source"].unique())

In [ ]:
df_l.info()

In [ ]:
df_final
